In [2]:
import os
os.chdir(r"G:\PycharmProjects\names-demo")

In [ ]:
import pandas as pd
import os
import polars as pl
# folder containing your files
folder_path = "data/raw/names-usa/namesbystate"

# list all files (adjust extension if needed)
files = [f for f in os.listdir(folder_path) if f.endswith(".TXT")]

df_list = []

for file in files:
    file_path = os.path.join(folder_path, file)
    df = pd.read_csv(file_path,header=None,names=["state", "gender", "year", "name", "count"])
    df_list.append(df)

# combine all files into one dataframe
df = pd.concat(df_list, ignore_index=True)
df.head()

In [ ]:
state_map = {
    'AK': 'Alaska', 'AL': 'Alabama', 'AR': 'Arkansas', 'AZ': 'Arizona',
    'CA': 'California', 'CO': 'Colorado', 'CT': 'Connecticut',
    'DC': 'District of Columbia', 'DE': 'Delaware', 'FL': 'Florida',
    'GA': 'Georgia', 'HI': 'Hawaii', 'IA': 'Iowa', 'ID': 'Idaho',
    'IL': 'Illinois', 'IN': 'Indiana', 'KS': 'Kansas', 'KY': 'Kentucky',
    'LA': 'Louisiana', 'MA': 'Massachusetts', 'MD': 'Maryland',
    'ME': 'Maine', 'MI': 'Michigan', 'MN': 'Minnesota',
    'MO': 'Missouri', 'MS': 'Mississippi', 'MT': 'Montana',
    'NC': 'North Carolina', 'ND': 'North Dakota', 'NE': 'Nebraska',
    'NH': 'New Hampshire', 'NJ': 'New Jersey', 'NM': 'New Mexico',
    'NV': 'Nevada', 'NY': 'New York', 'OH': 'Ohio', 'OK': 'Oklahoma',
    'OR': 'Oregon', 'PA': 'Pennsylvania', 'RI': 'Rhode Island',
    'SC': 'South Carolina', 'SD': 'South Dakota', 'TN': 'Tennessee',
    'TX': 'Texas', 'UT': 'Utah', 'VA': 'Virginia', 'VT': 'Vermont',
    'WA': 'Washington', 'WI': 'Wisconsin', 'WV': 'West Virginia',
    'WY': 'Wyoming'
}
df["state"] = df["state"].map(state_map)
# 'sex' sütunundaki değerleri yeni değerlerle eşleyelim
df.head()

In [ ]:
# Rename gender labels
df['gender'] = df['gender'].map({'F': 'female', 'M': 'male'})
# Generate gender based total counts
df['total_count'] = df.groupby(['state','gender', 'year'])['count'].transform('sum')
df.head()

In [49]:
# Create rank column
df['rank'] = df.groupby(['year', 'gender','state'])['count'].rank(ascending=False, method='min').astype(int)

# Sonuçları 'count' ve 'rank' sütunlarına göre sıralayarak kontrol edelim
df = df.sort_values(by=['year', 'gender', 'rank'])

In [50]:
df[(df["year"]==2024) & (df["gender"]=="female")].head(44)

,state,gender,year,name,count,total_count,rank
15509,Alaska,female,2024,Amelia,39,1652,1
108002,Alabama,female,2024,Charlotte,234,18655,1
233794,Arkansas,female,2024,Olivia,132,10489,1
356580,Arizona,female,2024,Olivia,346,27906,1
657438,California,female,2024,Mia,1986,164782,1
899467,Colorado,female,2024,Olivia,277,22644,1
1002341,Connecticut,female,2024,Mia,167,11353,1
1002342,Connecticut,female,2024,Olivia,167,11353,1
1072736,District of Columbia,female,2024,Charlotte,48,2271,1
1119189,Delaware,female,2024,Charlotte,46,2085,1


In [51]:
# Save to Parquet
df.to_parquet("data/preprocessed/usa/names_usa_states.parquet", engine="pyarrow", index=False)

In [9]:
# Read the saved file for checking
# import polars as pl
df = pl.read_parquet("data/preprocessed/usa/names_usa_states.parquet")
df.head()

state,gender,year,name,count,total_count
str,str,i64,str,i64,i64
"""Alaska""","""female""",1910,"""Mary""",14,352089
"""Alaska""","""female""",1910,"""Annie""",12,352089
"""Alaska""","""female""",1910,"""Anna""",10,352089
"""Alaska""","""female""",1910,"""Margaret""",8,352089
"""Alaska""","""female""",1910,"""Helen""",7,352089


In [35]:
import geopandas as gpd

url = "https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json"
gdf = gpd.read_file(url)[["name","geometry"]]
gdf.head()

,name,geometry
0,Alabama,"POLYGON ((-87.3593 35.00118, -85.60668 34.9847..."
1,Alaska,"MULTIPOLYGON (((-131.60202 55.11798, -131.5691..."
2,Arizona,"POLYGON ((-109.0425 37.00026, -109.04798 31.33..."
3,Arkansas,"POLYGON ((-94.47384 36.50186, -90.15254 36.496..."
4,California,"POLYGON ((-123.23326 42.00619, -122.37885 42.0..."


In [36]:
import geopandas as gpd
gdf.rename(columns={"name":"state"},inplace=True)
gdf.to_parquet("data/preprocessed/usa/usa_states_geo.parquet")

print(gdf.head())

        state                                           geometry
0     Alabama  POLYGON ((-87.3593 35.00118, -85.60668 34.9847...
1      Alaska  MULTIPOLYGON (((-131.60202 55.11798, -131.5691...
2     Arizona  POLYGON ((-109.0425 37.00026, -109.04798 31.33...
3    Arkansas  POLYGON ((-94.47384 36.50186, -90.15254 36.496...
4  California  POLYGON ((-123.23326 42.00619, -122.37885 42.0...


In [26]:
print(gdf.shape)

(50, 2)
